In [1]:
# 00_eda_eye_tracking.py
# EDA e merge dos dados de rastreamento ocular com metadados
# Alvos principais:
# - Ler e concatenar os 25 CSVs de eye-tracking
# - Limpar "Unidentified"
# - Merge com Metadata_Participants.csv pela coluna correta
# - Exportar merged_eye_tracking_metadata.csv para os próximos passos

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1) Caminhos
data_path = "../data/raw/Eye-tracking Output" # ajuste se necessário
metadata_path = "../data/raw/Metadata_Participants.csv"

# 2) Listar apenas CSVs dos experimentos (ignorar metadata.csv no diretório se houver)
eye_tracking_files = [
    f
    for f in os.listdir(data_path)
    if f.endswith(".csv") and f.lower() != "metadata.csv"
]
assert len(eye_tracking_files) > 0, f"Nenhum CSV encontrado em {data_path}"

# 3) Ler e concatenar
dfs = []
for file in eye_tracking_files:
    fp = os.path.join(data_path, file)
    df = pd.read_csv(fp, low_memory=False)
    dfs.append(df)
eye_data = pd.concat(dfs, ignore_index=True)

# 4) Remover 'Unidentified' em Participant (se a coluna existir)
if "Participant" in eye_data.columns:
    eye_data = eye_data[
        ~eye_data["Participant"].astype(str).str.contains("Unidentified", na=False)
    ]
else:
    raise KeyError(
        "Coluna 'Participant' não encontrada nos CSVs de eye-tracking. Verifique os arquivos."
    )

# 5) Ler metadata dos participantes
metadata = pd.read_csv(metadata_path, low_memory=False)
if "ParticipantID" not in metadata.columns:
    raise KeyError(
        "Coluna 'ParticipantID' não encontrada no metadata. Abra o CSV e confira os nomes."
    )

# 6) Converter IDs para numérico para merge robusto
eye_data["Participant"] = pd.to_numeric(eye_data["Participant"], errors="coerce")
metadata["ParticipantID"] = pd.to_numeric(metadata["ParticipantID"], errors="coerce")

# 7) Merge
merged_data = pd.merge(
    eye_data, metadata, how="left", left_on="Participant", right_on="ParticipantID"
)

# 8) Relatórios básicos
participants_classes = merged_data[["Participant", "Class"]].drop_duplicates()
print("Shape do DataFrame merged:", merged_data.shape)
print(
    "\nValores ausentes por coluna (top 20):\n",
    merged_data.isnull().sum().sort_values(ascending=False).head(20),
)
print("Participantes únicos (eye):", eye_data["Participant"].nunique())
print("IDs únicos no metadata:", metadata["ParticipantID"].nunique())
print(
    "Interseção (IDs):",
    len(set(eye_data["Participant"]).intersection(set(metadata["ParticipantID"]))),
)
print(
    "\nContagem por classe:\n", participants_classes["Class"].value_counts(dropna=False)
)

# 9) Gráficos (opcional - comente se estiver rodando em ambiente sem display)
try:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=participants_classes, x="Class")
    plt.title("Participantes por Classe")
    plt.tight_layout()
    plt.savefig("eda_participantes_por_classe.png", dpi=150)
    plt.close()
except Exception as e:
    print("Aviso: não foi possível renderizar gráficos (ambiente headless).", e)

# 10) Exportar para o próximo passo
merged_out = "merged_eye_tracking_metadata.csv"
merged_data.to_csv(merged_out, index=False)
print(f"✅ Arquivo salvo: {merged_out}")

Shape do DataFrame merged: (1351157, 64)

Valores ausentes por coluna (top 20):
 groupe d'enfants              1269387
AOI Group Left                 942996
AOI Scope Left                 942135
AOI Order Binocular            942135
AOI Group Right                878852
AOI Order Right                873255
AOI Scope Right                873255
CARS Score                     857767
Port Status                    706989
Pupil Size Left X [px]         669266
Pupil Size Left Y [px]         669266
Pupil Size Right Y [px]        600386
Pupil Size Right X [px]        600386
Pupil Diameter Left [mm]       445638
Pupil Diameter Right [mm]      376758
Eye Position Left X [mm]       292508
Eye Position Left Y [mm]       292508
Eye Position Left Z [mm]       292508
Pupil Position Left X [px]     292508
Pupil Position Left Y [px]     292508
dtype: int64
Participantes únicos (eye): 57
IDs únicos no metadata: 59
Interseção (IDs): 57

Contagem por classe:
 Class
TD     30
ASD    27
Name: count, dtype